# Тур 2 задача A. Острова рекомендаций

In [2]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

In [3]:
items = pd.read_csv('./items_A.csv')
also_viewed = pd.read_csv('./also_viewed_A.csv')

In [4]:
items.head()

,item_id,category,price,rating,brand,in_stock
0,1,laptops,161790,4.3,Alpha,1
1,2,toys,7612,4.5,ReadMore,1
2,3,home,39088,4.7,WorkPro,0
3,4,books,2808,3.2,Delta,1
4,5,accessories,3279,4.8,ReadMore,0


In [5]:
also_viewed.head()

,item_from,item_to
0,7,11
1,11,7
2,11,22
3,22,11
4,22,30


## 1

In [6]:
len(items[(items['category'] == 'phones') * (items['rating'] >= 4.5) * (items['in_stock'] == 1)])

22

## 2

In [26]:
rating = items[items['category'] == 'laptops'].groupby(by='brand').agg({'price': np.mean}).sort_values(by='price')
print(rating.iloc[-1])

price    165387.5
Name: Gamma, dtype: float64


## 3

In [8]:
len(items[(items['in_stock'] == 1) * (items['rating'] >= 4.5) * (items['price'] >= 50000)])

25

## 4

In [17]:
ans = pd.DataFrame()

ans['category'] = items.iloc[also_viewed['item_to'].tolist()]['category']
ans = ans.groupby(by='category').size().reset_index(name='cnt')
ans.to_csv("answer4.csv", index=False)

## 5

In [10]:
len(also_viewed)

758

In [11]:
adj_matrix = [[0] * len(also_viewed) for _ in range(len(also_viewed))]
adj_matrix = np.array(adj_matrix)

for i in range(len(also_viewed)):
    pair = also_viewed.iloc[i]
    adj_matrix[pair['item_from']][pair['item_to']] = 1
    adj_matrix[pair['item_to']][pair['item_from']] = 1

In [12]:
id_to_category = dict(zip(items['item_id'], items['category']))
cnt = 0
visited = [0] * len(adj_matrix)

def dfs(adj_mx, v) -> bool:
    visited[v] = 1
    has_phone = False
    
    if id_to_category.get(v) == 'phones':
        has_phone = True
        
    to = adj_mx[v]
    for u in range(len(to)):
        if to[u] == 1 and visited[u] == 0:
            if dfs(adj_mx, u):
                has_phone = True
    return has_phone

for v in range(len(visited)):
    if visited[v] == 0:
        if dfs(adj_matrix, v):
            cnt += 1

cnt

11

# Тур 1 задача H. Острова рекомендаций

### По аналогии с нормализацией кластера в K-Means

In [14]:
n = int(input())

inp = []
for _ in range(n):
    x, y, l = map(int, input().split())
    inp.append([x, y, l])
inp = np.array(inp)

xm = inp[:, 0][inp[:, 2] == -1].mean()
ym = inp[:, 1][inp[:, 2] == -1].mean()

def dist(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

rad = 0
for p in inp[inp[:, 2] == -1]:
    crad = dist(np.array(xm, ym), np.array(p[0], p[1]))
    if crad > rad:
        rad = crad

print(xm, ym, rad, sep=' ')

 0


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

# Тур 1 задача D. MAE

In [15]:
from sklearn.metrics import mean_absolute_error

X_ = np.array([[0], [0], [0], [0], [4], [4], [6], [6], [9], [9], [9], [9], [11], [11]])
y_ = np.array([0, 2, 7, 8, 4, 7, 1, 5, 0, 1, 4, 7, 2, 6])

class MyLinearRegression:
  def __init__(self, eps=1e-6, max_iter=10000, learning_rate=0.05):
    self.eps = eps
    self.max_iter = max_iter
    self.learning_rate = learning_rate

    self.w = None
    self.b = None

  def fit(self, X, y):
    X = np.hstack([np.ones((X.shape[0], 1)), X])
    self.w = np.zeros(X.shape[1])

    for _ in range(self.max_iter):
      grad = 1/len(X) * X.T @ np.sign(X @ self.w - y)

      if np.linalg.norm(grad) <= self.eps:
        break

      self.w -= self.learning_rate * grad

    self.b = self.w[0]
    self.w = self.w[1:]

    return self

  def predict(self, X):
    if self.w is None or self.b is None:
      raise ValueError("Fit model before predicting!")
    return X @ self.w + self.b

model = MyLinearRegression().fit(X_, y_)

In [16]:
mean_absolute_error(y_, model.predict(X_))

2.4285714285714284